# Sequence Prediction - STRICT protocol, WITHOUT app metadata

Copy of `_results_sequence_prediction_strict.ipynb` with the concatenated
app-metadata (SBERT) half removed: the projector + LSTM are trained *and*
evaluated on the Qwen screen embeddings alone.

Every stage still runs with `--strict`, i.e. the evaluation population
Screen2Vec itself uses in `modeltester.py`:

| | permissive (default notebook) | `--strict` (this notebook) |
|---|---|---|
| traces kept | >= 3 screens | >= `n_predictors + 1` (= 5) |
| context per prediction | 1..4 screens (truncated near trace start) | exactly 4 |
| candidate pool | every embedding in the directory | only screens of eligible traces |
| top-k% thresholds | `ceil()` | `int()` truncation, as in `modeltester.py:141` |
| training negatives | any screen | only screens of eligible traces |

Because every training window is full under `--strict`, the zero-padding branch
in the loss never fires, so training and evaluation see the same context length.

### What is removed, and why

`6_generate_app_context_embeddings.py` embeds each screen's Play-Store app
description with SBERT (`sentence-transformers/all-mpnet-base-v2`, 768-d) and
concatenates that vector onto the projected screen vector. It is a *per-app*
constant: every screen of the same app carries the identical second half.

| stage | sibling notebook (with app desc) | this notebook (`_wo_app`) |
|---|---|---|
| `9_train_joint.py` | 4096 -> `PROJ_DIM` projector + LSTM | **identical** - training never sees the app half |
| `6_generate_app_context_embeddings.py` | concatenates the 768-d SBERT app vector onto every screen vector | **not run** |
| `4_evaluate_screen_prediction.py` | retrieval over `PROJ_DIM + 768`-d vectors | retrieval over `PROJ_DIM`-d screen vectors |

The LSTM itself is unaffected by the removal, because it never consumed the app
half in the first place: `9_train_joint.py` trains on the raw Qwen embeddings
only, and at evaluation time `4_evaluate_screen_prediction.py` slices the screen
part off (`ctx_raw[:, :lstm_dim]`) before the forward pass. The app half enters
*only* in the ranking step, where it is copied verbatim from the **target** onto
the prediction (`4_evaluate_screen_prediction.py:373-375`). That is what these
notebooks remove: the retrieval query no longer carries the target's own app
identity.

Why it matters (see `SUMMARY.md`): with the app half attached, every candidate
from the target's app scores high, and since a RICO app has ~7.1 screens on
average (median 5), the Top-0.01% ... Top-5% columns largely measure app
identity rather than screen prediction - a measured app-only oracle already
reaches 0.9968 at Top-0.1% and 1.0 at Top-1%. In this notebook **every** column,
not just Top-1, is a screen-level number.

No code change is needed for any of this: when the embedding dimension equals
`lstm_dim`, the concatenation branch in the evaluator simply does not fire.

**Artefacts.** `9_train_joint.py` writes exactly the same files as the sibling
notebook, so the training cell below reuses them when they are already on disk.
Nothing is written under a `_with_app_desc` name and nothing here can clobber a
with-app run - the whole difference is which directory the evaluator is pointed
at.

    embeddings_seq_768_strict/                  (9_, shared with the strict sibling)
    lstm_predictor_768d_strict.pth              (9_, shared with the strict sibling)
    sequence_projector_768d_strict.pth          (9_, shared with the strict sibling)
    embeddings_seq_768_strict_with_app_desc/    (6_)  - NOT used here

### Caveats on the numbers this produces

1. **Screen ordering.** Screen2Vec builds each trace from an unsorted
   `os.listdir()` of `view_hierarchies`
   (`UI_embedding/precompute_embeddings.py:72`), which is not temporal order and
   is not reproducible on another filesystem. This pipeline always uses
   `gestures.json` order, i.e. true temporal order.

2. **Not comparable to the paper's own Top-k%.** Screen2Vec's Table 3 numbers
   were produced *with* the concatenated app-description half. Compare this
   notebook against its `_with_app_desc` sibling to read the size of that effect;
   do not put a `_wo_app` Top-1% next to a paper Top-1% and call it a
   reproduction.

3. **Trace and layout id namespaces are joined through the index.** Embeddings
   are keyed by Rico's *global* `data/layouts/` numbering while `gestures.json`
   keys are per-trace indices; every stage here resolves them through
   `trace_layout_index.csv` (`0_build_trace_index.py`) and refuses to run without
   it. Unmatched trace screens are dropped, so the evaluated population is the
   matched subset.


In [1]:
# ==== Configuration: strict protocol, no app metadata ====
PROJ_DIM = 768   # projector output dimension

EMB_DIR = f"embeddings_seq_{PROJ_DIM}_strict"          # 9_ output, evaluated as-is
LSTM_PATH = f"lstm_predictor_{PROJ_DIM}d_strict.pth"   # 9_ output
LABEL = ">>> Qwen3-VL + LSTM (Ours, no app desc) [strict] <<<"

print(f"protocol = strict  |  proj_dim = {PROJ_DIM}  |  app metadata = OFF")
print(f"  projected embeddings : {EMB_DIR}/")
print(f"  LSTM checkpoint      : {LSTM_PATH}")


protocol = strict  |  proj_dim = 768  |  app metadata = OFF
  projected embeddings : embeddings_seq_768_strict/
  LSTM checkpoint      : lstm_predictor_768d_strict.pth


In [2]:
# 9_train_joint.py is IDENTICAL to the with-app sibling notebook: it trains on the
# raw Qwen embeddings and never sees the app half, so it writes the very same
# artefacts. Reuse them when they are already on disk; delete one to force a retrain.
import os

if os.path.isfile(LSTM_PATH) and os.path.isdir(EMB_DIR):
    print(f"Reusing existing artefacts:\n  {LSTM_PATH}\n  {EMB_DIR}/")
    print("(delete either one to retrain from scratch)")
else:
    get_ipython().run_line_magic("run", f"9_train_joint.py --proj_dim {PROJ_DIM} --strict")


Reusing existing artefacts:
  lstm_predictor_768d_strict.pth
  embeddings_seq_768_strict/
(delete either one to retrain from scratch)


In [3]:
# 6_generate_app_context_embeddings.py is deliberately NOT run here. That stage is
# the only place the SBERT app-description vector enters the pipeline, so skipping
# it - and pointing the evaluator at EMB_DIR instead of EMB_DIR_with_app_desc - is
# the whole of the `_wo_app` variant.


In [4]:
%run 4_evaluate_screen_prediction.py --embeddings_dir $EMB_DIR --lstm_model $LSTM_PATH --strict --label "$LABEL"



[Step 0/3] Loading LSTM predictor from 'lstm_predictor_768d_strict.pth'...
  Loaded LSTM predictor (embed_dim=768, best_val_loss=2.5168)

[Step 1/3] Loading embeddings...
Loading 66261 embeddings from 'embeddings_seq_768_strict' in parallel...


Loading embeddings: 100%|██████████| 66261/66261 [00:58<00:00, 1137.57it/s]


  Loaded matrix shape: (66261, 768)
  66261 screen embeddings available

[Step 2/3] Parsing RICO interaction traces...
  Loaded trace->layout index: 66157 screens ('trace_layout_index.csv')


Parsing traces: 100%|██████████| 9384/9384 [00:15<00:00, 604.80it/s]


  Parsed 5095 traces (54520 screens)
  Skipped 5197 traces with < 5 screens
  Skipped 104 screens absent from the trace->layout index
  Strict pool: restricted to 54520 screens from eligible traces (was 66261)

[Step 3/3] Running screen prediction evaluation...

  Protocol: STRICT (Screen2Vec modeltester.py population, exactly 4 context screens per prediction)

  Total screens in embedding index: 54520
  Top-0.01% threshold: k=5
  Top-0.1%  threshold: k=54
  Top-1%    threshold: k=545
  Top-5%    threshold: k=2726
  Mean embedding L2 norm: 1.0000
  Prediction mode: LSTM (Screen2Vec-equivalent)
  Evaluating traces in parallel via threads (n_jobs=-1)...


Evaluating traces (LSTM): 100%|██████████| 5095/5095 [24:02<00:00,  3.53it/s]



  Total prediction instances: 34140

  GUI Screen Prediction — Comparison with Paper Table 3
Model                                            Top-1  Top-0.01%   Top-0.1%   Top-1%   Top-5%  Norm.RMSE
---------------------------------------------------------------------------------------------------------
>>> Qwen3-VL + LSTM (Ours, no app desc) [strict] <<<   0.0697     0.2443     0.6532   0.9159   0.9782     0.8525
---------------------------------------------------------------------------------------------------------
Screen2Vec-EUCLIDEAN-spatial                    0.0610     0.2580     0.9690   0.9980   1.0000     0.8530
Screen2Vec-EUCLIDEAN-no spatial                 0.0065     0.1160     0.8960   0.9860   0.9990     1.7230
TextOnly                                        0.0120     0.0550     0.1960   0.4390   0.6430     1.2410
LayoutOnly                                      0.0041     0.0240     0.0910   0.2220   0.3950     1.1350
VisualOnly                                      0.0